<span style="color: #E6E6E6;">
<span style="color: #E6E6E6; background-color: #00695C; padding:8px; display:block; border-left:4px solid #4682b4">

# Attention

*In this exercise we are going to work with attention. If you haven't heard, it has become **massively** important, mostly because transformers use it (and it is part of what makes them the new black for everything).*

After these exercises (and the lecture), you should be able to (in rough order of importance):
- Understand what attention is
- Understand how attention is computed given inputs
- Understand why it is often called 'self-attention'
- Understand how neural networks are used to get attention values
- Understand why attention is good to use
- Understand how attention differs from just embeddings
- Understand early (non-learnable) vs newer learnable attention

**Note**: This notebook contains some code taken or adapted from 02456: Deep Learning, accessed from the [GitHub repo](https://github.com/DeepLearningDTU/02456-deep-learning-with-PyTorch) on 16-11-2024. Some code snippets may differ from earlier in the course, as there hasn't been time to adapt them entirely to the course code structure.

## Quick introduction

*In its most basic form, attention is a similarity measure applied to sequences, usually of words. As you may know, the dot product is a rough measure of similarity between vectors. Let $\mathbf{x}^{(1)}, \dots, \mathbf{x}^{(N)}$ be the word embeddings of a sentence of $N$ words, for example a sentence in English, and let $\mathbf{y}^{(1)}, \dots, \mathbf{y}^{(M)}$ be the word embeddings of a sentence of $M$ words, for example the same sentence in another language. The similarity (the **attention score**) between word $i$ in the first sentence and word $j$ in the second sentence is then:*

$$e_{ij} = \mathbf{x}^{(i)T} \mathbf{y}^{(j)}$$

*That is, how much word $\mathbf{x}^{(i)}$ relates to word $\mathbf{y}^{(j)}$. The softmax is then used to scale these scores to be between 0 and 1. It is taken over $j$ (i.e. over all words in the second sentence), so that the weights for word $i$ sum to 1. This gives the **attention weight** $W_{ij}$ between word $i$ and word $j$:*

$$W_{ij} = \text{softmax}_j\left(e_{ij}\right) = \frac{\exp\left(e_{ij}\right)}{\sum_{k=1}^{M} \exp\left(e_{ik}\right)}$$

*After this, the **attention weights** $W_{ij}$ are used to take a weighted sum of the words in the second sentence, which gives the final **attended representations**:*

$$\mathbf{o}^{(i)} = \sum_{j=1}^{M} W_{ij}\, \mathbf{y}^{(j)}$$

*The result $\mathbf{o}^{(i)}$ will then be **a vector** (with the same dimension $d$ as the word embeddings) that summarizes the second sentence as seen from **one word** $i$ of the first sentence: the words in the second sentence that are most similar to word $i$ contribute the most. The attention weights **from each word** to **each word** form a matrix $W$ of size $(N, M)$, and stacking all the $\mathbf{o}^{(i)}$ gives a matrix $\mathbf{O}$ of size $(N, d)$.*

*If the two sequences are the **same** sentence, i.e. the sentence attends to itself, it is called **self-attention**.*

## Improved (learnable) attention

*Introduced primarily in [Attention Is All You Need](https://arxiv.org/abs/1706.03762), the transformer improves 'regular' attention by adding learnable parameters to it.*

*Note that above, the attention values only depended on the similarity between words, meaning that words with similar embeddings got high attention scores. This doesn't make sense for many tasks, particularly translation, since some words in a sentence can relate a lot to each other despite being different. Consider the sentence:*

***Mary saw a dog in the window, she wanted it***

*Logically, we can think of the words (Mary, saw, wanted, it), (dog, it) and (in, window) as fitting together, as they describe the relationship that Mary wanted **it**, where "it" means "the dog" (not the window). This would not be caught by the above method, since "it" and "dog" are not very similar words...*

*The solution is to compute three new vectors for each word: a **query**, a **key** and a **value**. Stacked for the whole sentence, they form the matrices $\mathbf{Q}$ (queries), $\mathbf{K}$ (keys) and $\mathbf{V}$ (values). Do not worry too much about the names.*

*These are not new word embeddings themselves. **They work on embeddings**. To get $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ for one input sentence (of words!!) $\mathbf{s}$, we first get a word embedding of that input sentence (a matrix with one row per word), and then multiply that matrix with a learnable weight matrix for each of $\mathbf{Q}, \mathbf{K}, \mathbf{V}$:*

$$\mathbf{Q} = \text{embedding}(\mathbf{s})\, W_Q,\quad \mathbf{K} = \text{embedding}(\mathbf{s})\, W_K, \quad \mathbf{V} = \text{embedding}(\mathbf{s})\, W_V$$

*The weight matrices $W_Q, W_K, W_V$ are trained with backpropagation together with the rest of the network, on whatever task the network is trained for (e.g. translation). Since $\mathbf{Q}$, $\mathbf{K}$ and $\mathbf{V}$ all come from the same sentence, this is still **self-attention**, but now the model can **learn** that "it" should attend to "dog", even though the two words are not similar.*

*We then compute the attention weights (the softmax part, taken over each row) and use them to take a weighted sum of the values:*

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left( \frac{\mathbf{Q} \mathbf{K}^T}{\sqrt{d_k}} \right) \mathbf{V}$$

*Here $d_k$ is the dimension of the keys (for multi-head attention, the key dimension of each head). We scale by $\sqrt{d_k}$ because dot products between long vectors get large (their variance grows with $d_k$). Large scores make the softmax put almost all the weight on a single word, where its gradients are close to zero, so the model learns very slowly (don't worry too much about it)...*

## Difference between attention **weights** and attended **representations**

*Note that we differentiate between the **attention weights** $W$ and the **attended representations** $\mathbf{O}$. The attention weights are often the ones examined to find out what a model thinks about a specific sentence, since they form a matrix of dimension $(N, M)$, i.e. (sentence 1 length, sentence 2 length), with one value per pair of words. The attended representations, however, are of dimension $(N, d)$, i.e. (sentence 1 length, embedding dim), making them much harder to make sense of.*

</span>
</span>


In [ ]:
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn
import torch.nn.functional as F
import zipfile
import warnings

from pathlib import Path
from typing import List, Optional, Tuple, Union

# NOTE: You may have to do some 'uv pip install' here (tokenizers, huggingface_hub)
import tokenizers

from huggingface_hub import hf_hub_download


In [ ]:
# The following are just helper functions to get GloVe vectors and such
# NOTE: The first time you run this, it downloads the GloVe zip file (~860 MB) from Hugging Face.
#       It is cached afterwards, so later runs are fast.

def load_glove_vectors(filename = "glove.6B.50d.txt") -> Tuple[List[str], torch.Tensor]:
    """Download and load GloVe vectors, returns both vocabulary and GloVe vectors"""
    try:
        path = Path(hf_hub_download(repo_id="stanfordnlp/glove", filename="glove.6B.zip"))
    except Exception as e:
        warnings.warn(f"Failed to download GloVe vectors from Hugging Face Hub: {e}. Please check your internet connection and try again.", UserWarning)
        raise
    target_file = path.parent / filename
    if not target_file.exists():
        with zipfile.ZipFile(path, 'r') as zip_ref:
            zip_ref.extractall(path.parent)

        if not target_file.exists():
            print(f"Available files:")
            for p in path.parent.iterdir():
                print(p)
            raise ValueError(f"Target file `{target_file.name}` can't be found. Check if `{filename}` was properly downloaded.")

    # parse the vocabulary and the vectors
    vocabulary = []
    vectors = []
    with open(target_file, "r", encoding="utf8") as f:
        for l in tqdm(f.readlines(), desc=f"Parsing {target_file.name}..." ):
            word, *vector = l.split()
            vocabulary.append(word)
            vectors.append(torch.tensor([float(v) for v in vector]))
    vectors = torch.stack(vectors)
    return vocabulary, vectors

# Load GloVe, and add a token for unknown words (with an all-zero vector),
# so words that are not in GloVe don't make the tokenizer crash
glove_vocabulary, glove_vectors = load_glove_vectors()
glove_vocabulary.append("<|unknown|>")
glove_vectors = torch.cat([glove_vectors, torch.zeros(1, glove_vectors.shape[1])])

# Tokenizer for GloVe - simplifies a lot of things,
# such as when passing to transformers or when handling unknown tokens
glove_tokenizer = tokenizers.Tokenizer(tokenizers.models.WordLevel(vocab={v:i for i,v in enumerate(glove_vocabulary)}, unk_token="<|unknown|>"))
glove_tokenizer.normalizer = tokenizers.normalizers.BertNormalizer(strip_accents=False)
glove_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()

embeddings = torch.nn.Embedding(*glove_vectors.shape)
embeddings.weight.data = glove_vectors
embeddings.weight.requires_grad = False

def words_to_glove(sentence: Union[str, List[str]]) -> torch.Tensor:
    """
    Slightly more sophisticated method of getting GloVe vectors

    Words that the tokenizer splits into several tokens (e.g. 'rug!' -> 'rug', '!') get the mean of their token vectors

    Note: Returns torch tensors!
    """

    # Just handle in case user forgot that input needs to be a list
    if isinstance(sentence, str):
        sentence = sentence.split()

    encodings = glove_tokenizer.encode_batch(sentence)
    vectors = [embeddings(torch.tensor(e.ids)).mean(dim=0) for e in encodings]
    return torch.stack(vectors)

# Test the function works
sentence_example = 'snug as bug in a rug!'
print(words_to_glove(sentence_example).shape)

<span style="color: #E6E6E6; background-color: #545454; padding:8px; display:block; border-left:4px solid #4682b4">

### Exercise 1 - 💻 Calculating attention

*Here we are going to use a few mockup examples of sentences with GloVe embeddings to generate attention values between sentences.*

**1. Complete the function below to calculate just the simple attention weights between the word vectors of two sentences. That is, we want the attention from each word in `sentence_1` to each word in `sentence_2`**

**2. In general, given one sentence of $N$ words and a second sentence of $M$ words, what dimension should the attention from the first sentence to the second sentence have?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

**3. Complete the function to get the attended representations using the previously calculated attention weights**

**4. Reason about what the attention weights vs. the attended representations can be used for...**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

</span>

In [ ]:
def simple_attention_weights(vec_1, vec_2):
    """
    Computes simple attention scores between two sentences

    Returns an N X M attention map where N, M are the lengths of sentence 1 and 2 respectively
    """

    # Initialize empty attention map
    attention_map = ...

    # For each word (vector) in sentence 1
    for i in range(len(vec_1)):
        # Iterate over all word (vectors) in sentence 2
        for j in range(len(vec_2)):
            # And compute the attention score by dot product
            ...

    # Apply softmax to each row of the attention map (i.e. over the words in sentence 2) to get final attention
    # Tip: subtracting the row maximum before np.exp does not change the result, but avoids overflow
    attention_map = ...

    return attention_map

def attended_representation(vec_1, vec_2, attention_weights):
    attended_representations = np.zeros_like(vec_1)
    for i in range(len(vec_1)):
        for j in range(len(vec_2)):
            ...

    return attended_representations


In [ ]:
# Two example sentences
sentence_1 = 'a cat in a hat'
sentence_2 = 'a bug in a rug hat'

# Get the word vectors, we un-torch them here, otherwise we'll run into issues with numpy vs torch
vec_1 = words_to_glove(sentence_1).detach().numpy()
vec_2 = words_to_glove(sentence_2).detach().numpy()

# Get the attention weights
attention_weights = simple_attention_weights(vec_1, vec_2)

print("Dimensionality of attention weights is:", attention_weights.shape)
print("Attention weights are:", attention_weights)


In [ ]:
# Get the attended representations, no reason to print anything other than their dimension
attended_representations = attended_representation(vec_1, vec_2, attention_weights=attention_weights)

print("Dimensionality of attended_representations is:", attended_representations.shape)

<span style="color: #E6E6E6; background-color: #545454; padding:8px; display:block; border-left:4px solid #4682b4">

### Exercise 2 - Visualizing attention

*Often we want to see how much each word in one sentence attends to each word in another. Ours is quite simple, since it only relies on word embeddings, but there may still be cool things to uncover...*

**1. Run the two code cells below to plot the attention weights for two new sentences (the plot uses your `simple_attention_weights` function from before). Where is attention large? Where is it small, and why?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

**2. Is there potentially a scaling issue when using attention in this manner? (HINT: Why does 'a'-'a' have a smaller attention value than 'snug'-'snug'?)**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

**$\star$ 3. Reason a bit about how you *could* use this attention mechanism for NLP purposes**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

</span>

In [ ]:
def visualize_attention(attention_map, sentence_1: List[str], sentence_2: List[str], head: Optional[int] = None):
    """
    Visualize the attention map between two sentences.

    Parameters:
        attention_map (numpy.ndarray): 2D array with attention weights. Shape: (len(sentence_1), len(sentence_2))
            (or 3D with shape (num_heads, len(sentence_1), len(sentence_2)) if head is given)
        sentence_1 (list of str): The first sentence, tokenized into words.
        sentence_2 (list of str): The second sentence, tokenized into words.
        head (int, optional): Which attention head to plot, for multi-head attention.
    """

    if head is not None:
        attention_map = attention_map[head]

    # Ensure the attention map dimensions match the sentences
    assert attention_map.shape == (len(sentence_1), len(sentence_2)), (
        "Attention map shape must match sentence lengths"
    )

    # Width follows sentence 2 (x-axis), height follows sentence 1 (y-axis)
    fig, ax = plt.subplots(figsize=(max(4, len(sentence_2)), max(4, len(sentence_1))))
    cax = ax.matshow(attention_map, cmap='viridis', aspect='auto')

    # Add color bar
    plt.colorbar(cax, ax=ax, orientation='vertical')

    # Set up ticks and labels
    ax.set_xticks(np.arange(len(sentence_2)))
    ax.set_yticks(np.arange(len(sentence_1)))
    ax.set_xticklabels(sentence_2, rotation=90, fontsize=10)
    ax.set_yticklabels(sentence_1, fontsize=10)

    # Add labels for clarity
    ax.set_xlabel('Words in Sentence 2', fontsize=12)
    ax.set_ylabel('Words in Sentence 1', fontsize=12)
    plt.title('Attention Map', fontsize=14)

    if head is not None:
        plt.title(f'Attention Map for head {head}', fontsize=14)


    plt.tight_layout()
    plt.show()


In [ ]:
# Test attention map function with new example sentences
sentence_1 = 'snug like a cat in a hat'
sentence_2 = 'snug as a bug in a rug'

vec_1 = words_to_glove(sentence_1).detach().numpy()
vec_2 = words_to_glove(sentence_2).detach().numpy()

attention_map = simple_attention_weights(vec_1, vec_2)
visualize_attention(attention_map, sentence_1.split(), sentence_2.split())

<span style="color: #E6E6E6; background-color: #545454; padding:8px; display:block; border-left:4px solid #4682b4">

### Exercise 3 - 💻 Creating a vocabulary

*NLP usually requires a ton of preprocessing, which we really cannot be bothered to do. However, there aren't many places in the program to practice it...*

**1. $\star$ $\star$ $\star$ Do the thing we cannot be bothered to do: perform some of the preprocessing by completing the function to build a vocabulary from a list of sentences**

**If you do not want to do it, just copy the `build_vocabulary` function from the solutions**

</span>

In [ ]:
# Build a vocabulary, so that every word gets an integer index
# We need these indices for torch's embedding class in Exercise 4, since it is just a bit closer to what you'd normally use

def build_vocabulary(sentences, oov_token='OOV'):

    # Make a set to represent our vocabulary
    # A set is like a dict with keys, but no values
    # Really more like a list that cannot have duplicates
    vocab = set()

    # Iterate through sentences, lowercase and split the sentences to obtain lists of words used, add these to vocab
    ...

    # Add the only special token we use: OOV (out of vocabulary)
    vocab.update([oov_token])

    # Sort vocab, just to make it nicer to look around in
    vocab = sorted(vocab)

    # Create the dictionaries word_to_index (word -> index) and index_to_word (index -> word)
    word_to_index = ...
    index_to_word = ...

    # Define function for getting token indices from a list of tokens, should return the index of OOV if a word is not in vocab
    def tokens_to_indices(tokens):
        return ...

    return tokens_to_indices, word_to_index, index_to_word, vocab

# Define completely random sentences to have a vocabulary
sentences = [
    "He turned himself into a pickle",
    "Sphinx of Black Quartz, Judge My Vow",
    "Skibidi dum dum yes yes",
    "The brain is rotting on the poles",
    "Release me from this flesh prison mortal fool",
    "I submit myself willingly to our artificial overlords",
    "One day the crude biomass that you call a temple will wither and you will beg my kind to save you",
    "For everything to be consummated for me to feel less alone I had only to wish that there be a large crowd of spectators the day of my execution and that they greet me with cries of hate"
]

# Create vocab based on sentences:
tokens_to_indices, word_to_index, index_to_word, vocab = build_vocabulary(sentences=sentences)

# Create a new sentence by randomly sampling from vocab...
# (seeded, so everyone gets the same sentence - feel free to change the seed)
np.random.seed(42)
new_sentence = " ".join(np.random.choice(vocab, 5))
# Add extra word just to showcase OOV tokens also being used
new_sentence += " factorio"
print("Random sentence is", new_sentence)

# Get tokens from the new sentence:
tokens = new_sentence.lower().split()
print("Tokens are", tokens)
token_indices = tokens_to_indices(tokens)

print("Token indices are", token_indices)

print([index_to_word[i] for i in token_indices])


<span style="color: #E6E6E6; background-color: #545454; padding:8px; display:block; border-left:4px solid #4682b4">

### Exercise 4 - 💻 The actual attention part

*The 'groundbreaking' paper [Attention Is All You Need](https://arxiv.org/abs/1706.03762) originally proposed using matrices of queries, keys and values, along with multi-head attention. We will attempt to do the same, albeit without training an actual model.*

*The reason we don't train a model is that it is really time consuming, and NLP requires tons of preprocessing that we (as mentioned) really cannot be bothered to do right now...*

*Here we are going to build a simple attention module using torch that does not train, and simply uses randomly initialized weights.*

**1. Complete the `AttentionModule` class (the layers in `__init__` and the `attention_from_tokens` function). Test that it works by running the first code cell after the class definitions. Remember, the attention should have shape `(sentence_length, sentence_length)` and the representations should have shape `(sentence_length, V_dim)`**

**2. Does the attention make sense? Does it hold any kind of interesting information whatsoever? Why/Why not?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

**3. Could we expect better results by using pretrained GloVe as our word embeddings while keeping the Q, K and V matrices the same? Why/Why not?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

**4. Complete the code to implement multi-head attention, and test it by running the second code cell after the class definitions. Below are a few suggestions on how you can do this (there is no single correct way!):**

1. Keep a list of query, key and value layers (one of each per head), and do the same calculation as for a single head in a for loop or list comprehension
2. Scale the output dimensions of the query, key and value layers by the number of heads. You can then reshape the resulting tensors to get `(num_heads, sentence_length, V_dim)`, and pass the result to a linear layer or aggregation function

- NOTE: In either case, you should implement a linear layer to aggregate the results of all heads into a matrix of shape `(sentence_length, V_dim)`
- NOTE: Be careful when reshaping! A tensor of shape `(sentence_length, num_heads * Q_dim)` should first be viewed as `(sentence_length, num_heads, Q_dim)` and then have its first two dimensions swapped. Viewing it directly as `(num_heads, sentence_length, Q_dim)` mixes up words and heads
- NOTE: The softmax should be taken over the **keys** (the last dimension), so each row of each head's attention map sums to 1

Feel free to take inspiration from the solution, but try to understand it.

**5. What are the advantages of using a linear layer to aggregate the results of all attention heads, compared to just using like the mean or something?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

</span>


In [ ]:
class AttentionModule:
    def __init__(self, word_dim, Q_dim, K_dim, V_dim):
        # Create dummy embedding layer for word embeddings (one for each word in the vocab from Exercise 3):
        # A torch embedding layer is like a linear layer that takes integer indices (technically torch.long values) instead of one-hot vectors
        self.word_embeddings = ...

        # Define weight matrices for Queries, Keys and Values
        # Unlike the word embeddings, these work on the (float) embedding vectors rather than on integer indices,
        # so they are just linear layers, despite being used much like embeddings
        self.query_embeddings = ...
        self.key_embeddings = ...
        self.value_embeddings = ...

        # Keep track of K_dim for the scaling factor
        self.K_dim = K_dim

    def attention_from_tokens(self, token_indices):
        # Transform to tensor, so it can be used in word_embeddings:
        token_indices_tensor = torch.tensor(token_indices)
        embeddings = self.word_embeddings(token_indices_tensor)

        attention = ...

        representations = ...

        return attention, representations

# Just define Multi head attention as subclass of Attention module
class MultiHeadAttentionModule(AttentionModule):
    def __init__(self, word_dim, Q_dim, K_dim, V_dim, num_heads=1):
        super().__init__(word_dim, Q_dim, K_dim, V_dim)

        # Difference from Attention Module, actually has a number of heads
        self.num_heads = num_heads

        # This time, we make the query, key and value layers num_heads times bigger (replacing the single-head ones from super().__init__)
        # Technically we should have a [num_heads, word_dim, Q_dim] dimensional weight tensor, but torch's linear layers only use 2-dimensional weight matrices...
        # ... so we just increase the number of Q_dims and then we can reshape afterwards
        self.query_embeddings = ...
        self.key_embeddings = ...
        self.value_embeddings = ...

        # This time, we actually need all of our Q, K and V dims for reshaping purposes...
        self.Q_dim = Q_dim
        self.K_dim = K_dim
        self.V_dim = V_dim

        # Final linear layer to combine the results of all heads into one...
        self.linear_layer = ...

    def attention_from_tokens(self, token_indices):
        # Transform to tensor, so it can be used in word_embeddings:
        token_indices_tensor = torch.tensor(token_indices)
        embeddings = self.word_embeddings(token_indices_tensor)
        sentence_length = len(token_indices)

        # Then we can get our multi-head queries, keys and values, each of shape (num_heads, sentence_length, Q/K/V_dim)
        queries = ...
        keys = ...
        values = ...

        # Get our queries dotted with our keys
        queries_dot_keys = ...

        # Get the multi-head attention and multi-head representations
        heads_attention = ...
        heads_representations = ...

        # Finally, pass through linear layer as a kind of aggregation...
        final_representations = ...

        return heads_attention, final_representations


In [ ]:
# Define dimensions of word embeddings, as well as Q, K and V matrices
word_dim = 50
Q_dim = 50
K_dim = 50
V_dim = 50

# Instantiate AttentionModule
attention_model = AttentionModule(word_dim, Q_dim, K_dim, V_dim)

# Print sentence we are getting attention for
print("Sentence:", new_sentence)

# Calculate the attention weights and the attended representations
attention, representations = attention_model.attention_from_tokens(token_indices)

# Print and visualize
print("Attention weights have shape:", attention.shape)
print("Representations (attention weights times values) have shape:", representations.shape)
print("Each row of the attention sums to 1:", torch.allclose(attention.sum(dim=-1), torch.ones(len(token_indices))))

visualize_attention(attention.detach().numpy(), new_sentence.split(), new_sentence.split())

In [ ]:
num_heads = 3

# Instantiate MultiHeadAttentionModule
multihead_attention_model = MultiHeadAttentionModule(word_dim, Q_dim, K_dim, V_dim, num_heads=num_heads)


# Calculate the attention weights and the final representations
multihead_attention, multihead_representations = multihead_attention_model.attention_from_tokens(token_indices)

# Print and visualize
print("Attention weights have shape:", multihead_attention.shape)
print("Final representations (after the linear layer) have shape:", multihead_representations.shape)
print("Each row of each head's attention sums to 1:", torch.allclose(multihead_attention.sum(dim=-1), torch.ones(num_heads, len(token_indices))))
visualize_attention(multihead_attention.detach().numpy(), new_sentence.split(), new_sentence.split(), head=0)

<span style="color: #E6E6E6; background-color: #545454; padding:8px; display:block; border-left:4px solid #4682b4">

### Exercise 5 - More in-depth view of attention

*While trying to make some really cool representation of attention, I almost forgot the creed of the KID student... But then I remembered, and found [this cool visualization notebook](https://colab.research.google.com/drive/1hXIQ77A4TYS4y3UthWF-Ci7V7vVUoxmQ?usp=sharing) in [the bertviz repository](https://github.com/jessevig/bertviz). It uses a pretrained BERT transformer **with** positional encodings (next week's topic). It also has multiple attention layers, so it doesn't just produce one attention map.*

**1. Look around the provided notebook and see if you get any smarter about attention mechanisms. Remember, BERT has multiple heads across multiple layers that each produce attention!**

- NOTE: You do not need to run anything, the visualizations are already there, preloaded

**$\star$ $\star$ $\star$ $\star$ $\star$ $\star$ $\star$ 2. Say something nice to the person next to you. How did they respond?**

<span style="color: #E6E6E6; background-color: #00590D; padding:8px; display:block; border-left:4px solid #4682b4">

Your answer here $\dots$

</span>

</span>